In [68]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
import xgboost as xgb
import lightgbm as lgb

In [73]:
sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 110

RAW_PATH = r'C:\Users\amare\OneDrive\Desktop\Projects\Finance Hackathon\stdbank-pip\data\01_raw\Synthetic_Financial_Datasets_For_Fraud_Detection_resample.csv'
TOL = 0.01

In [74]:
df = pd.read_csv(r'C:\Users\amare\OneDrive\Desktop\Projects\Finance Hackathon\stdbank-pip\data\01_raw\Synthetic_Financial_Datasets_For_Fraud_Detection_resample.csv')

In [75]:
df

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,182,CASH_IN,100328.70,C197678862,3389602.37,3489931.07,C386191921,219735.14,119406.44,0,0
1,210,PAYMENT,11660.99,C2009511954,94054.00,82393.01,M564284134,0.00,0.00,0,0
2,403,CASH_IN,230751.30,C1132312861,23117012.10,23347763.39,C1025694734,951294.81,720543.51,0,0
3,328,PAYMENT,17833.60,C1191709365,0.00,0.00,M293024801,0.00,0.00,0,0
4,563,CASH_OUT,246476.67,C342438889,0.00,0.00,C2087488957,4226850.16,4473326.82,0,0
...,...,...,...,...,...,...,...,...,...,...,...
636257,325,CASH_OUT,193812.50,C425986045,95438.00,0.00,C1813421380,511348.22,705160.72,0,0
636258,324,PAYMENT,71240.51,C246016542,5021.00,0.00,M332903748,0.00,0.00,0,0
636259,231,CASH_IN,219467.05,C152812322,3990.00,223457.05,C316565040,2593190.49,2373723.44,0,0
636260,306,PAYMENT,1048.54,C37754145,0.00,0.00,M1903551785,0.00,0.00,0,0


In [76]:

TOLERANCE = 0.01

orig_ok = np.isclose(df['newbalanceOrig'],
                     df['oldbalanceOrg'] - df['amount'],
                     atol=TOLERANCE)

dest_ok = np.isclose(df['newbalanceDest'],
                     df['oldbalanceDest'] + df['amount'],
                     atol=TOLERANCE)

# Keep only rows where BOTH sides of the ledger add up
filtered = df[orig_ok & dest_ok].copy()

print(f'Original rows : {len(df):,}')
print(f'Filtered rows : {len(filtered):,}')

Original rows : 636,262
Filtered rows : 27,990


In [81]:
filtered['isFraud'].value_counts()

isFraud
0    27589
1      401
Name: count, dtype: int64

In [63]:
df[df['nameDest']=='C1049817027']

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,hour_of_day,day_of_week
963,1,TRANSFER,411363.54,C1200048933,39951.00,0.00,C1049817027,52170.00,60738.03,0,0,1,0
1128,1,CASH_OUT,151096.79,C977997405,0.00,0.00,C1049817027,463533.54,60738.03,0,0,1,0
1327,1,CASH_IN,414285.51,C1318475957,8918271.71,9332557.22,C1049817027,614630.33,60738.03,0,0,1,0
2048,1,CASH_OUT,32282.57,C123674777,14375.74,0.00,C1049817027,97055.04,60738.03,0,0,1,0
2182,1,CASH_IN,25927.54,C1457417579,8151115.24,8177042.78,C1049817027,129337.61,60738.03,0,0,1,0
2183,1,CASH_IN,42672.03,C1500697171,8177042.78,8219714.81,C1049817027,103410.07,60738.03,0,0,1,0
3941,3,CASH_IN,58930.33,C745835029,4358882.39,4417812.72,C1049817027,60738.03,1807.70,0,0,3,0
70901,9,CASH_OUT,47452.09,C542905872,12057.00,0.00,C1049817027,1807.70,49259.79,0,0,9,0
104189,10,CASH_IN,36611.71,C764539279,3751802.53,3788414.24,C1049817027,49259.79,12648.08,0,0,10,0
242765,14,CASH_IN,246652.81,C347024574,14787.00,261439.81,C1049817027,12648.08,0.00,0,0,14,0


P(oldbalanceOrg == amount | fraud) = 0.9753P(oldbalanceOrg == amount | legit) = 0.0000Mechanism: the fraudster empties the account (amount = full balance), andPaySim cancels fraudulent transactions, so newbalanceOrig stays untouched.A single boolean feature therefore separates the classes almost perfectly.This is why the public leaderboard for this dataset sits at AUC 0.99+.

In [67]:
df[(df["oldbalanceOrg"] == df["amount"]) & (df["isFraud"] == 1)]

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
2,1,TRANSFER,181.00,C1305486145,181.00,0.0,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.0,C38997010,21182.00,0.00,1,0
251,1,TRANSFER,2806.00,C1420196421,2806.00,0.0,C972765878,0.00,0.00,1,0
252,1,CASH_OUT,2806.00,C2101527076,2806.00,0.0,C1007251739,26202.00,0.00,1,0
680,1,TRANSFER,20128.00,C137533655,20128.00,0.0,C1848415041,0.00,0.00,1,0
...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.0,C776919290,0.00,339682.13,1,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.0,C1881841831,0.00,0.00,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.0,C1365125890,68488.84,6379898.11,1,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.0,C2080388513,0.00,0.00,1,0


# Every fraudulent transaction where the sender empties their account is either a TRANSFER or CASH_OUT

In [47]:
df1["type"].unique()

array(['TRANSFER', 'CASH_OUT'], dtype=object)

In [27]:
# Calculate the expected destination balance after the transfer
df['expected_newbalanceDest'] = df['oldbalanceDest'] + df['amount']

# Calculate the absolute discrepancy error at the destination
df['dest_balance_error'] = df['expected_newbalanceDest'] - df['newbalanceDest']

In [37]:
def engineer_fraud_features(df: pd.DataFrame) -> pd.DataFrame:
    """Engineers balance error, anomaly flags, and behavioral features

    for detecting transaction fraud on the dataset.
    """
    # Create a copy to avoid SettingWithCopyWarning
    df = df.copy()

    # =========================================================================
    # 1. Base Ledger Discrepancy Features (Original + Cleaned)
    # =========================================================================

    # Expected new balance at destination after receiving funds
    df["expected_newbalanceDest"] = df["oldbalanceDest"] + df["amount"]

    # Difference between expected destination balance and actual recorded balance
    # Clean tiny floating-point artifacts (e.g., -4.65e-10 -> 0.0)
    df["dest_balance_error"] = (
        df["expected_newbalanceDest"] - df["newbalanceDest"]
    ).round(2)

    # Expected new balance at origin after sending funds
    df["expected_newbalanceOrig"] = df["oldbalanceOrg"] - df["amount"]

    # Difference between expected origin balance and actual recorded balance
    df["orig_balance_error"] = (
        df["expected_newbalanceOrig"] - df["newbalanceOrig"]
    ).round(2)

    # =========================================================================
    # 2. Account Behavioral Patterns
    # =========================================================================

    # Flag transactions where the sender empties their full account balance
    df["is_account_emptied"] = (df["oldbalanceOrg"] == df["amount"]).astype(int)

    # =========================================================================
    # 3. Destination Inconsistency Anomalies
    # =========================================================================

    # Binary flag for transactions where money moved, but destination balance remained 0
    df["is_zero_dest_balance"] = (
        (df["oldbalanceDest"] == 0) & (df["newbalanceDest"] == 0)
    ).astype(int)

    # Binary flag where the destination discrepancy equals the full transaction amount
    df["error_equals_amount"] = (
        df["dest_balance_error"].abs() == df["amount"].round(2)
    ).astype(int)

    # =========================================================================
    # 4. Interaction Features (Type + Anomalies)
    # =========================================================================

    # Specific flag for suspicious TRANSFER transactions with zero destination balance
    df["is_transfer_zero_dest"] = (
        (df["type"] == "TRANSFER") & (df["is_zero_dest_balance"] == 1)
    ).astype(int)

    # Flag for CASH_OUT transactions that contain unexpected balance discrepancies
    df["is_cashout_with_error"] = (
        (df["type"] == "CASH_OUT") & (df["dest_balance_error"].abs() > 0.01)
    ).astype(int)

    return df


# Example usage:
df_featured = engineer_fraud_features(df)

In [38]:
df_featured

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,expected_newbalanceDest,dest_balance_error,expected_newbalanceOrig,orig_balance_error,is_account_emptied,is_zero_dest_balance,error_equals_amount,is_transfer_zero_dest,is_cashout_with_error
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0,9839.64,9839.64,160296.36,0.0,0,1,1,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0,1864.28,1864.28,19384.72,0.0,0,1,1,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0,181.00,181.00,0.00,0.0,1,1,1,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0,21363.00,21363.00,0.00,0.0,1,0,0,0,1
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0,11668.14,11668.14,29885.86,0.0,0,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6362615,743,CASH_OUT,339682.13,C786484425,339682.13,0.00,C776919290,0.00,339682.13,1,0,339682.13,0.00,0.00,0.0,1,0,0,0,0
6362616,743,TRANSFER,6311409.28,C1529008245,6311409.28,0.00,C1881841831,0.00,0.00,1,0,6311409.28,6311409.28,0.00,0.0,1,1,1,1,0
6362617,743,CASH_OUT,6311409.28,C1162922333,6311409.28,0.00,C1365125890,68488.84,6379898.11,1,0,6379898.12,0.01,0.00,0.0,1,0,0,0,0
6362618,743,TRANSFER,850002.52,C1685995037,850002.52,0.00,C2080388513,0.00,0.00,1,0,850002.52,850002.52,0.00,0.0,1,1,1,1,0


In [42]:
# Hour of the day (0 to 23)
df['hour_of_day'] = df['step'] % 24

# Day of the week (0 to 6)
df['day_of_week'] = (df['step'] // 24) % 7

In [43]:
# Drastically cuts dataset size without losing any positive fraud cases
df_model = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()

In [49]:
df_model[["isFraud"]].value_counts(normalize=True)

isFraud
0          0.997035
1          0.002965
Name: proportion, dtype: float64

In [50]:
df_model.shape

(2770409, 13)

In [26]:
df[(df["oldbalanceOrg"] == df["amount"]) & (df["isFraud"] == 0)]

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud


In [22]:
df["nameOrig"].duplicated().sum()

np.int64(9313)

In [23]:
df["nameDest"].duplicated().sum()

np.int64(3640258)

In [21]:
df[df["nameOrig"] == df["nameDest"]]

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
